In [1]:
import muon as mu
import numpy as np
import pandas as pd
import anndata

DATA_PATH = '/home/lemanic-g04/'

mdata = mu.read(DATA_PATH + 'GSE194315_binarized_mu.h5mu')
protein = mdata.mod['protein']

/home/gnoto_venvs/text/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/gnoto_venvs/text/lib/python3.12/site-packages/muon/_core/preproc.py:31: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1272: Fut

In [2]:
# Build text
X = protein.X.toarray() if hasattr(protein.X, 'toarray') else protein.X
X = X.astype(bool)
protein_names = protein.var_names.to_numpy()

def humanize(names):
    names = list(names)
    if len(names) == 0:
        return ""
    if len(names) == 1:
        return names[0]
    if len(names) == 2:
        return f"{names[0]} and {names[1]}"
    return ", ".join(names[:-1]) + f", and {names[-1]}"

def cell_to_text(row):
    expressed = protein_names[row]
    not_expressed = protein_names[~row]

    if len(expressed) == 0:
        return (
            f"This cell does not express any of the measured surface proteins. "
            f"Specifically, it lacks expression of {humanize(not_expressed)}."
        )
    if len(not_expressed) == 0:
        return (
            f"This cell expresses all of the measured surface proteins. "
            f"Specifically, it expresses {humanize(expressed)}."
        )
    return (
        f"This cell expresses the following surface proteins: {humanize(expressed)}. "
        f"However, it does not express {humanize(not_expressed)}."
    )

texts = [cell_to_text(X[i]) for i in range(X.shape[0])]


In [4]:
# Preview
for i in range(3):
    print(f"--- {protein.obs_names[i]} ---")
    print(texts[i])
    print()

# Save
mdata.mod['protein'].obs['text'] = texts
mdata.obs['text'] = texts

anndata.settings.allow_write_nullable_strings = True
out_path = DATA_PATH + 'GSE194315_binarized_with_text_mu.h5mu'
mdata.write(out_path)
print(f"Saved to {out_path}")

pd.DataFrame({'cell_id': protein.obs_names, 'text': texts}).to_csv(
    'cell_texts.csv', index=False
)
print("Also saved text_conversion/cell_texts.csv")

--- PBMC-02-1_AAACCCAGTAGCTGTT ---
This cell expresses the following surface proteins: CD352-NTB-A, GPR56, KLRG1-MAFA, TCRVdelta2, CD2-1, CD3, and CD45RA. However, it does not express C5L2, Cadherin11, CCR10-1, CD101-BB27, CD102-ICAM-2, CD105, CD106, CD107a-LAMP-1, CD109-1, CD110, CD112-Nectin-2, CD115-CSF-1R, CD116, CD119-IFN-gammaRalphachain, CD131, CD137L-4-1BBLigand, CD138-Syndecan-1, CD140a-PDGFRalpha, CD140b-PDGFRbeta, CD142, CD144-VE-Cadherin, CD146, CD15-SSEA-1, CD150-SLAM, CD151-PETA-3, CD158d-KIR2DL4, CD158e1-KIR3DL1-NKB1, CD158f-KIR2DL5, CD161, CD162, CD164-1, CD169-Sialoadhesin-Siglec-1, CD171-LICAM, CD172a-SIRPalpha, CD177-1, CD179a-VpreB, CD186-CXCR6, CD192-CCR2, CD193-CCR3, CD198-CCR8, CD199-CCR9, CD1d, CD200-OX2, CD201-EPCR, CD203c-E-NPP3, CD204, CD205-DEC-205, CD207-Langerin, CD209-DC-SIGN, CD215-IL-15Ralpha, CD218a-IL-18Ralpha, CD22-1, CD23, CD230-Prion, CD235ab, CD243-ABCB1, CD244-2B4, CD252-OX40L, CD253-Trail, CD254-TRANCE-RANKL, CD257-BAFF-BLYS, CD267-TACI, CD268-B

/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/gnoto_venvs/text/lib/python3.12/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


Saved to /home/lemanic-g04/GSE194315_binarized_with_text_mu.h5mu
Also saved text_conversion/cell_texts.csv


In [5]:
# Verify text matches the binarized matrix for all cells
all_ok = True
mismatches = []

for i in range(len(texts)):
  row = X[i]
  true_pos = set(protein_names[row])
  true_neg = set(protein_names[~row])
  text = texts[i]

  # Parse the three possible templates
  if "However, it does not express" in text:
      pos_part, neg_part = text.split("However, it does not express", 1)
      pos_part = pos_part.replace("This cell expresses the following surface proteins:", "").strip().rstrip(".")
      neg_part = neg_part.strip().rstrip(".")
  elif "does not express any of the measured" in text:
      pos_part = ""
      neg_part = text.split("it lacks expression of", 1)[1].strip().rstrip(".")
  elif "expresses all of the measured" in text:
      pos_part = text.split("it expresses", 1)[1].strip().rstrip(".")
      neg_part = ""
  else:
      mismatches.append((i, "unrecognized template"))
      all_ok = False
      continue

  def parse_list(s):
      if not s:
          return set()
      s = s.replace(", and ", ", ").replace(" and ", ", ")
      return set(p.strip() for p in s.split(",") if p.strip())

  text_pos = parse_list(pos_part)
  text_neg = parse_list(neg_part)

  if text_pos != true_pos or text_neg != true_neg:
      mismatches.append((i, protein.obs_names[i]))
      all_ok = False

print(f"Checked {len(texts)} cells")
print(f"Mismatches: {len(mismatches)}")
if mismatches:
  print("First 5 mismatches:")
  for m in mismatches[:5]:
      print(f"  {m}")
print("All good" if all_ok else "Problems found — inspect mismatches above")

Checked 180794 cells
Mismatches: 0
All good


In [6]:
suspect = [n for n in protein_names if ', and ' in n or ' and ' in n]
print("Problem names:", suspect)

Problem names: []


In [7]:
# Manual inspection for one specific cell
barcode = "PBMC-02-1_AAACCCAGTAGCTGTT"

i = list(protein.obs_names).index(barcode)
row = X[i]

print(f"=== {barcode} (cell index {i}) ===\n")

print(f"Expressed ({row.sum()} proteins):")
for name in protein_names[row]:
  print(f"  1  {name}")

print(f"\nNot expressed ({(~row).sum()} proteins):")
for name in protein_names[~row]:
  print(f"  0  {name}")

print(f"\n=== Generated text ===\n")
print(texts[i])

=== PBMC-02-1_AAACCCAGTAGCTGTT (cell index 0) ===

Expressed (7 proteins):
  1  CD352-NTB-A
  1  GPR56
  1  KLRG1-MAFA
  1  TCRVdelta2
  1  CD2-1
  1  CD3
  1  CD45RA

Not expressed (250 proteins):
  0  C5L2
  0  Cadherin11
  0  CCR10-1
  0  CD101-BB27
  0  CD102-ICAM-2
  0  CD105
  0  CD106
  0  CD107a-LAMP-1
  0  CD109-1
  0  CD110
  0  CD112-Nectin-2
  0  CD115-CSF-1R
  0  CD116
  0  CD119-IFN-gammaRalphachain
  0  CD131
  0  CD137L-4-1BBLigand
  0  CD138-Syndecan-1
  0  CD140a-PDGFRalpha
  0  CD140b-PDGFRbeta
  0  CD142
  0  CD144-VE-Cadherin
  0  CD146
  0  CD15-SSEA-1
  0  CD150-SLAM
  0  CD151-PETA-3
  0  CD158d-KIR2DL4
  0  CD158e1-KIR3DL1-NKB1
  0  CD158f-KIR2DL5
  0  CD161
  0  CD162
  0  CD164-1
  0  CD169-Sialoadhesin-Siglec-1
  0  CD171-LICAM
  0  CD172a-SIRPalpha
  0  CD177-1
  0  CD179a-VpreB
  0  CD186-CXCR6
  0  CD192-CCR2
  0  CD193-CCR3
  0  CD198-CCR8
  0  CD199-CCR9
  0  CD1d
  0  CD200-OX2
  0  CD201-EPCR
  0  CD203c-E-NPP3
  0  CD204
  0  CD205-DEC-205
  0  CD207